# Teaching Strategy Experiment

Last updated: 2026-07-13 14:00 JST

This notebook checks the teacher-strategy layer. It generates one student utterance, lets the communication AI observe it, builds a teacher context, and selects the next teaching strategy.

## 1. Pull latest code from GitHub

Run this cell at the top of Colab. It clones the repo if needed, otherwise it pulls the latest code.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Settings

Keep `USE_MOCK_MODEL = True` for a quick check. Set it to `False` when you want a real local LLM to generate the student utterance.

In [ ]:
STUDENT_ID = "S002"
USE_MOCK_MODEL = True
TEACHER_MESSAGE = "Solve x + 3 = 8. When you move +3 to the other side, what happens to the sign?"

print("student_id:", STUDENT_ID)
print("use_mock_model:", USE_MOCK_MODEL)
print("teacher_message:", TEACHER_MESSAGE)

## 4. Generate student utterance

This uses lesson dialogue mode. Knowledge updates are off here so the experiment is easy to repeat.

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=USE_MOCK_MODEL)
student_record = sim.respond(
    STUDENT_ID,
    TEACHER_MESSAGE,
    update_knowledge=False,
)
student_utterance = student_record["answer"]

print(student_utterance)

## 5. Communication AI observes the utterance

The communication AI estimates visible traits from the utterance and prepares hints for the teacher AI.

In [ ]:
from pprint import pprint

from src.observer import CommunicationAI

communication_ai = CommunicationAI()
observation = communication_ai.classify_utterance(
    utterance=student_utterance,
    student_id=STUDENT_ID,
).to_dict()

pprint(observation)

## 6. Build teacher context

The teacher context combines the student state, curriculum goal, recent utterance, communication AI observation, and next-problem bank.

In [ ]:
from pprint import pprint

from src.state_manager import StateManager
from src.teacher import TeacherContextBuilder

state_manager = StateManager()
student_state = state_manager.load_student(STUDENT_ID)

context_builder = TeacherContextBuilder()
teacher_context = context_builder.build_context(
    student_state=student_state,
    recent_student_utterance=student_utterance,
    communication_observation=observation,
)

pprint({
    "target_skill": teacher_context["target_skill"],
    "lesson_goal": teacher_context["lesson_goal"],
    "student_state_summary": teacher_context["student_state_summary"],
    "communication_ai_observation": teacher_context["communication_ai_observation"],
})

## 7. Select teaching strategy

The current MVP uses deterministic rules so the decision can be inspected. Later, the same context can be passed to an LLM teacher.

In [ ]:
from pprint import pprint

from src.teacher import RuleBasedTeachingStrategySelector

selector = RuleBasedTeachingStrategySelector()
teacher_decision = selector.select_strategy(teacher_context)

pprint(teacher_decision)

## 8. Save latest result

The latest teacher context and decision are saved as JSON files for later comparison.

In [ ]:
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)

context_path = output_dir / "teacher_context_latest.json"
decision_path = output_dir / "teaching_strategy_decision_latest.json"

context_path.write_text(json.dumps(teacher_context, ensure_ascii=False, indent=2), encoding="utf-8")
decision_path.write_text(json.dumps(teacher_decision, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", context_path)
print("saved:", decision_path)